In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 4.2 Final Model Selection and Deployment
- Selecting a model based on institutional priorities
- Saving and loading models
- Model card documentation

## Model Selection Criteria
| Factor | Question |
|:-------|:---------|
| Performance | Meets minimum thresholds? |
| Interpretability | Can advisors understand it? |
| Fairness | Equitable across subgroups? |
| Maintenance | Can team retrain it? |
| Compliance | Can we explain if audited? |

## Setup

In [ ]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

train_df = pd.read_csv('../data/training.csv')
test_df = pd.read_csv('../data/testing.csv')
train_df['DEPARTED'] = (train_df['SEM_3_STATUS'] != 'E').astype(int)
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

numeric_features = ['HS_GPA','HS_MATH_GPA','HS_ENGL_GPA','UNITS_ATTEMPTED_1','UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1','UNITS_COMPLETED_2','DFW_UNITS_1','DFW_UNITS_2','GPA_1','GPA_2',
    'DFW_RATE_1','DFW_RATE_2','GRADE_POINTS_1','GRADE_POINTS_2']
categorical_features = ['RACE_ETHNICITY','GENDER','FIRST_GEN_STATUS','COLLEGE']

train_enc = pd.get_dummies(train_df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
test_enc = pd.get_dummies(test_df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)
train_enc = train_enc.fillna(train_enc.median())
test_enc = test_enc.fillna(test_enc.median())

X_train, y_train = train_enc, train_df['DEPARTED']
X_test, y_test = test_enc, test_df['DEPARTED']
print("Data prepared.")

## Saving and Loading Models
```python
import joblib
joblib.dump(model, 'model.pkl')          # Save
loaded = joblib.load('model.pkl')        # Load
predictions = loaded.predict(new_data)   # Use
```

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=5,
    class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

joblib.dump(rf, '../models/random_forest_final.pkl')
print("Model saved.")

loaded_rf = joblib.load('../models/random_forest_final.pkl')
prob_original = rf.predict_proba(X_test)[:, 1]
prob_loaded = loaded_rf.predict_proba(X_test)[:, 1]
print(f"Predictions match: {np.allclose(prob_original, prob_loaded)}")
print(f"Test AUC: {roc_auc_score(y_test, prob_loaded):.4f}")

## Model Card Template
| Field | Description |
|:------|:------------|
| Model Name | Student Departure Risk Predictor v1.0 |
| Task | Binary classification: 3rd semester departure |
| Training Data | CSULB first-time freshmen cohorts |
| Features | HS GPA, college GPA, DFW rates, demographics |
| Intended Use | Early warning system for advisors |
| Limitations | Trained on CSULB data only |
| Ethical | Monitor for bias across demographic groups |
| Retraining | Annually with new cohort |

## Key Decisions
1. **Logistic Regression** when interpretability matters
2. **Random Forest** for reliable default
3. **XGBoost** for max accuracy
4. **Document** with a model card
5. **Plan for retraining** annually

**Module 4 Complete! Next: Module 5 -- Unsupervised Learning**